<a href="https://colab.research.google.com/github/Davide537/Internship-code/blob/main/Copy_of_LAOML_P1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Question 1
For this question we first imported the Numpy and Pandas libraries into python to help in processing the data. We then load the relevant data file into gooogle colab for analysis and perform some preliminary checks to make sure the file has loaded properly.



In [ ]:
import pandas as pd
import numpy as np
file_path = "data1.csv"
data = pd.read_csv(file_path)
data.head()

,android.permission.GET_ACCOUNTS,com.sonyericsson.home.permission.BROADCAST_BADGE,android.permission.READ_PROFILE,android.permission.MANAGE_ACCOUNTS,android.permission.WRITE_SYNC_SETTINGS,android.permission.READ_EXTERNAL_STORAGE,android.permission.RECEIVE_SMS,com.android.launcher.permission.READ_SETTINGS,android.permission.WRITE_SETTINGS,com.google.android.providers.gsf.permission.READ_GSERVICES,...,com.android.launcher.permission.UNINSTALL_SHORTCUT,com.sec.android.iap.permission.BILLING,com.htc.launcher.permission.UPDATE_SHORTCUT,com.sec.android.provider.badge.permission.WRITE,android.permission.ACCESS_NETWORK_STATE,com.google.android.finsky.permission.BIND_GET_INSTALL_REFERRER_SERVICE,com.huawei.android.launcher.permission.READ_SETTINGS,android.permission.READ_SMS,android.permission.PROCESS_INCOMING_CALLS,Result
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0


In [ ]:
X = data.iloc[:, :-1].values
y = data.iloc[:, -1]
y = np.where(y == 1, 1, -1)

num_malicious = np.sum(y == 1)
total_elements = X.size
zero_elements = np.sum(X == 0)
sparsity = zero_elements / total_elements

print(f"Number of malicious data points: {num_malicious}")
print(f"Sparsity of the data: {sparsity:.2%}")

Number of malicious data points: 14700
Sparsity of the data: 89.01%



We have 14700 malicious data points here.

Sparsity measures how often a feature (permission) is not requested by an application.
Here, the sparsity is high (almost 90%), meaning that a lot of permissions are not requested by the android applications. We can infer from this that storing the matrices in sparse format may be more efficient for our matrix vector multiplications.

One-hot encoding is a technique where categorical variables are transformed into binary columns, with each column representing one possible category.
We can therefore represent all the information with a matrix made up of 0s and 1s, with the objects in rows and the object characteristics in columns.

In our case, the matrix X represents the android applications in the rows, and the features (permissions) in the columns. Our features are binary indicators (0 or 1).
We can conclude that our matrix X is already in a one-hot encoding form, so we can't do further transformation.

# Question 2
Here we split the data into training and test sets. We decided to first train the model on 80% of the data and test it on the remaining 20%. We play around with this split later on in the assignment. In this question before we split the data into training and test sets we first use the numpy library to get a random permutation of the data. This ensures that the training sets are representative of the true dataset and prevents any bias or overfitting that may occur if the data is in an ordered structure.


In [ ]:
import numpy as np

def split_data(X, y, r=0.8):
    num_data_points = X.shape[0]
    indices = np.random.permutation(num_data_points)
    num_train = int(r * num_data_points)

    train_indices = indices[:num_train]
    test_indices = indices[num_train:]

    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]

    return X_train, X_test, y_train, y_test

###
X_train, X_test, y_train, y_test = split_data(X, y, r=0.8)
# print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(f"the data is split into sets of size {X_train.shape} and {y_train.shape} for the training set and of size {X_test.shape} and {y_test.shape} for the testing set")
# print(X_train, X_test, y_train, y_test)

the data is split into sets of size (23465, 86) and (23465,) for the training set and of size (5867, 86) and (5867,) for the testing set


# Question 3

Since we use $sign(x^Tw)$ to classify the vector $x$,a vector $x_i$ from matrix $X$ is correctly classified $\iff$ $y_ix_i^Tw>0$
As a sanity check we take W equal to the random weight vector, with entries normally distributed with mean 1.

In [ ]:
def corr_class_points_attempt1(X,y,w):
  counter=0

  for i in range(X.shape[0]):
    if y[i]*(X[i]@(w))>0:
      counter+=1

  return counter , counter/X.shape[0]

def corr_class_points(X,y,w):
  yXw = y * (X.dot(w))
  return (yXw > 0).sum() , (yXw > 0).sum()/X.shape[0]



for i in range(10):
  w = np.random.randn(X.shape[1])
  corr_points, percent_corr_points = corr_class_points(X,y,w)
  print(corr_points)
  print(f"There are {  corr_points} correctly identified points, that is {percent_corr_points:.2%}%")


14481
There are 14481 correctly identified points, that is 49.37%%
8789
There are 8789 correctly identified points, that is 29.96%%
12118
There are 12118 correctly identified points, that is 41.31%%
14630
There are 14630 correctly identified points, that is 49.88%%
16223
There are 16223 correctly identified points, that is 55.31%%
15086
There are 15086 correctly identified points, that is 51.43%%
14274
There are 14274 correctly identified points, that is 48.66%%
18774
There are 18774 correctly identified points, that is 64.01%%
7335
There are 7335 correctly identified points, that is 25.01%%
13802
There are 13802 correctly identified points, that is 47.05%%


We first wrote the function to count the number of correctly classified code in a "naive" way using for-loops. We implemented this function in the gradient descent but it made the algorithm inefficient.\
However, we noticed that we can write the expression $y_ix_i^Tw$ in a more compact way: $y_ix_i^Tw$ is the $i$-th row of the vector $Xw$ multiplied by the $i$-th entry of $y$. \\
This can be coded in python as $y * ( X.dot(w) )$. We then count the entries of $y * ( X.dot(w) )$ which have positive value. This is what we did in "corr_class_points". This gives a s As you can see below, this gives a more "polished" code that is also faster (see tests below).
We used this idea in other code that computes $y_ix_i^Tw$ such as the gradient "grad" and the sparse version of the gradient descent "sparse_gradient_descent" as well.


In [ ]:
import time


loops = 20
print(f"Taking an average of {loops} iterations:")

mean=np.zeros(loops)
for i in range(loops):
  w = np.random.randn(X.shape[1])
  start_time = time.time()
  corr_class_points_attempt1(X,y,w)
  end_time = time.time()
  mean[i]=(end_time - start_time)

mean=mean.sum()/loops
print(f"The first attempt at the code takes an average of {mean:.5f} seconds.")

mean=np.zeros(loops)
for i in range(loops):
  w = np.random.randn(X.shape[1])
  start_time = time.time()
  corr_class_points(X,y,w)
  end_time = time.time()
  mean[i]=(end_time - start_time)

mean=mean.sum()/loops
print(f"The second attempt at the code takes an average of {mean:.5f} seconds.")


Taking an average of 20 iterations:
The first attempt at the code takes an average of 0.19300 seconds.
The second attempt at the code takes an average of 0.00604 seconds.


# Question 4

The cost function for logistic regression is $$J(w)=\sum_{i=1}^{n}L(y_ix_i^Tw)+\frac{λ}{2}||w||^2$$
where $$L(s)=\log(1+e^{-s}).$$
$$∇L(s)=\frac{-e^{-s}}{1+e^{-s}}=\frac{-1}{e^s+1}$$
Hence $$∇ J(w)=\sum_{i=1}^{n}\frac{-y_i}{e^{y_ix_i^Tw}+1}x_i+λw$$

# Question 5

Gradient descent with fixed step size:$$w^{i+1}=w^{i}-\alpha \nabla J(w^i)$$

In [ ]:
def grad(X,y,l,w):
  n_samples, n_features = X.shape
  yXw = y * (X.dot(w))
  scalar = -y / (np.exp(yXw) + 1)
  result = (scalar[:, np.newaxis]* X).sum(axis=0)
  return result+l*w


def gradient_descent(X,y,alp,l,K):
  n_samples, n_features = X.shape
  w = np.zeros(n_features)

  for i in range(K):
    gradient = grad(X,y,l,w)
    w -= alp * gradient
  return w


w=gradient_descent(X_train,y_train,0.0001,0.1,100)
corr_points, percent_corr_points = corr_class_points(X_test,y_test,w)
print(f"There are {  corr_points} correctly identified points, that is {percent_corr_points:.2%}")

There are 5554 correctly identified points, that is 94.67%


The hyperparamters are important as too high a learning rate will cause the model to converge too quickly, causing us to miss the optimal point, called overshooting. Too low a regulization parameter can lead to overfitting, a phenomenmon where the models become too sensitive to new data. Also, the number of iterations is important as too few will not give the model enough time to converge to the optimal point, too many may be unnecessary and lead to a waste of compute/energy.


In [ ]:
from scipy.sparse import csr_matrix, csc_matrix
X_csr = csr_matrix(X)
X_csc = csc_matrix(X)


def sparse_gradient_descent(X, y, alp, l, K):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)

    for i in range(K):
        yXw = y * (X.dot(w))
        scalar = -y / (np.exp(yXw) + 1)
        sum=scalar*X
        gradient =  sum + l* w
        w -= alp * gradient
    return w


# sparse_gradient_descent(X_csr,y,0.0001,0.1,10)

In [ ]:
import time
from scipy.sparse import csr_matrix, csc_matrix


loops = 10
print(f"Taking an average of {loops} iterations:")
mean=np.zeros(loops)
for i in range(loops):
  start_time = time.time()
  gradient_descent(X,y,0.0001,0.1,100)
  end_time = time.time()
  mean[i]=(end_time - start_time)

mean=mean.sum()/loops
print(f"Dense matrix multiplication took an average of {mean:.5f} seconds.")

X_csr = csr_matrix(X)

mean=np.zeros(loops)
for i in range(loops):
  start_time = time.time()
  sparse_gradient_descent(X_csr,y,0.0001,0.1,100)
  end_time = time.time()
  mean[i]=(end_time - start_time)

mean=mean.sum()/loops
print(f"Compressed sparce row multiplication took an average of {mean:.5f} seconds.")

X_csc = csc_matrix(X)

mean=np.zeros(loops)
for i in range(loops):
  start_time = time.time()
  sparse_gradient_descent(X_csc,y,0.0001,0.1,100)
  end_time = time.time()
  mean[i]=(end_time - start_time)

mean=mean.sum()/loops
print(f"Compressed sparce column multiplication took an average of {mean:.5f} seconds.")



Taking an average of 5 iterations:
Dense matrix multiplication took an average of 1.66311 seconds.
Compressed sparce row multiplication took an average of 0.32074 seconds.
Compressed sparce column multiplication took an average of 0.43272 seconds.


We found that gradient desecent for the dense matrix operations takes on average about 2s while gradient descent with the sparse matrix formats takes a fraction of that time, with compressed sparse row operations taking about 0.6s and compressed sparse column taking 1s.
As expected, taking advanced of the sparsity of the matrix allows for faster code. Using CSR matrices is faster than using CSC matrices as the CSR format is more efficient at matrix-vector multiplication.

Of course the exact numbers may change if the code is run multiple times or if it is run on different hardware. Nonetheless the trend should remain the same.


# playing around with the split in the training data


In [ ]:
X_csc_train, X_csc_test, y_train, y_test = split_data(X_csc, y, r=0.5)
w=sparse_gradient_descent(X_csc_train,y_train,0.0001,0.1,100)
print(corr_class_points(X_csc_test,y_test,w))

(13816, 0.9420428201281876)


In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, r=0.5)
w=gradient_descent(X_train,y_train,0.0001,0.1,100)
print(corr_class_points(X_test,y_test,w))

(13877, 0.9462021000954589)


We obtain a slightly poorer score on the correctly classified points for the sparse matrix regression however, this is to be expected as we only retained the top k=54 singular values corresponding to 95% of the variance in the dataset.

Despite the worse performance we continue with the sparse format for the remainder of our computations due to its efficiency.

In [ ]:
X_csc_train, X_csc_test, y_train, y_test = split_data(X_csc, y, r=0.5)
w=sparse_gradient_descent(X_csc_train,y_train,0.0001,0.2,1000)
print(corr_class_points(X_csc_test,y_test,w))

(13951, 0.9512477839901814)


From adjusting the hyperparameters we found that 1000 iterations marginally improved the classification score compared to 100 iterations.  For a 50/50 split betwen training and test data with our original hyperparamters we obtain a correct classification ~95.1% of the time.

In [ ]:
X_csc_train, X_csc_test, y_train, y_test = split_data(X_csc, y, r=0.6)
w=sparse_gradient_descent(X_csc_train,y_train,0.0001,0.1,1000)
print(corr_class_points(X_csc_test,y_test,w))

(11189, 0.9536350464501833)


Playing with the value of $α$

In [ ]:
import matplotlib
X_csr = csr_matrix(X)
for i in range(-2,10):
  X_csr_train, X_csr_test, y_train, y_test = split_data(X_csr, y, r=0.5)
  print(f"The value of α is {10**(-i)}")
  w=sparse_gradient_descent(X_csr_train,y_train,10**(-i),0.01,100)
  corr_points, corr_points_percent = corr_class_points(X_csr_test,y_test,w)
  print(f"There are {corr_points} correctly classified points, representing {corr_points_percent:.2%} of all points")
  print("")



The value of α is 100
There are 7262 correctly classified points, representing 49.52% of all points

The value of α is 10


<ipython-input-7-9d8cd80e3685>:12: RuntimeWarning: overflow encountered in exp
  scalar = -y / (np.exp(yXw) + 1)


There are 13595 correctly classified points, representing 92.70% of all points

The value of α is 1
There are 13612 correctly classified points, representing 92.81% of all points

The value of α is 0.1
There are 13748 correctly classified points, representing 93.74% of all points

The value of α is 0.01
There are 13948 correctly classified points, representing 95.10% of all points

The value of α is 0.001
There are 13950 correctly classified points, representing 95.12% of all points

The value of α is 0.0001
There are 13866 correctly classified points, representing 94.55% of all points

The value of α is 1e-05
There are 13678 correctly classified points, representing 93.26% of all points

The value of α is 1e-06
There are 12298 correctly classified points, representing 83.85% of all points

The value of α is 1e-07
There are 10145 correctly classified points, representing 69.17% of all points

The value of α is 1e-08
There are 9093 correctly classified points, representing 62.00% of all

This shows empirically that taking $α=0.01$ gives more accuracy.

Playing with the value of $\lambda$

In [ ]:
X_csr = csr_matrix(X)
for i in range(-2,10):
  X_csr_train, X_csr_test, y_train, y_test = split_data(X_csr, y, r=0.5)
  print(f"The value of λ is {10**(-i)}")
  w=sparse_gradient_descent(X_csr_train,y_train,0.01, 10**(-i) ,100)
  corr_points, corr_points_percent = corr_class_points(X_csr_test,y_test,w)
  print(f"There are {corr_points} correctly classified points, representing {corr_points_percent:.2%} of all points")
  print("")


The value of λ is 100
There are 7192 correctly classified points, representing 49.04% of all points

The value of λ is 10


<ipython-input-7-9d8cd80e3685>:12: RuntimeWarning: overflow encountered in exp
  scalar = -y / (np.exp(yXw) + 1)


There are 13065 correctly classified points, representing 89.08% of all points

The value of λ is 1
There are 13855 correctly classified points, representing 94.47% of all points

The value of λ is 0.1
There are 13977 correctly classified points, representing 95.30% of all points

The value of λ is 0.01
There are 13935 correctly classified points, representing 95.02% of all points

The value of λ is 0.001
There are 13909 correctly classified points, representing 94.84% of all points

The value of λ is 0.0001
There are 13889 correctly classified points, representing 94.70% of all points

The value of λ is 1e-05
There are 13930 correctly classified points, representing 94.98% of all points

The value of λ is 1e-06
There are 13897 correctly classified points, representing 94.76% of all points

The value of λ is 1e-07
There are 13920 correctly classified points, representing 94.91% of all points

The value of λ is 1e-08
There are 13880 correctly classified points, representing 94.64% of al

From this, we can see that for $λ<1$, the accuracy of the method does not change much.

Playing with the splits

In [ ]:
X_csr = csr_matrix(X)
for i in range(10):
  X_csr_train, X_csr_test, y_train, y_test = split_data(X_csr, y, r=0.1*i)
  print(f"The value of r is {0.1*i:.0%}")
  w=sparse_gradient_descent(X_csr_train,y_train,0.01, 0.001 ,100)
  corr_points, corr_points_percent = corr_class_points(X_csr_test,y_test,w)
  print(f"There are {corr_points} correctly classified points, representing {corr_points_percent:.2%} of all points")
  print("")


The value of r is 0%
There are 0 correctly classified points, representing 0.00% of all points

The value of r is 10%
There are 24979 correctly classified points, representing 94.62% of all points

The value of r is 20%
There are 22318 correctly classified points, representing 95.11% of all points

The value of r is 30%
There are 19508 correctly classified points, representing 95.01% of all points

The value of r is 40%
There are 16716 correctly classified points, representing 94.98% of all points

The value of r is 50%
There are 13912 correctly classified points, representing 94.86% of all points

The value of r is 60%
There are 11128 correctly classified points, representing 94.84% of all points

The value of r is 70%


<ipython-input-7-9d8cd80e3685>:12: RuntimeWarning: overflow encountered in exp
  scalar = -y / (np.exp(yXw) + 1)


There are 8361 correctly classified points, representing 95.01% of all points

The value of r is 80%
There are 5553 correctly classified points, representing 94.65% of all points

The value of r is 90%
There are 2783 correctly classified points, representing 94.85% of all points



Highest accuracy is around 0.6-0.4 training to test ratio but it doesnt change much

In [ ]:
X_csr = csr_matrix(X)
X_csr_train, X_csr_test, y_train, y_test = split_data(X_csr, y, r=0.5)
w=sparse_gradient_descent(X_csr_train,y_train,0.01, 0.001 ,100)
corr_points, corr_points_percent = corr_class_points(X_csr_test,y_test,w)
print(f"There are {corr_points} correctly classified points, representing {corr_points_percent:.2%} of all points")

There are 13901 correctly classified points, representing 94.78% of all points


#Question 6
Here, we perform singular value decomposition on our new dataset. We choose to keep the first k=54 singular values which corresponded to capturing 95% of the variance in our dataset (2 standard deviations). In this example the data is centered as to make sure that principal compenents reflect the true *variability* around the data's mean rather than capturing the datas average position in space.

In [ ]:
from scipy.linalg import svd
data = pd.read_csv("data2.csv")

centered_data = data - data.mean()


U, s, Vt = svd(centered_data, full_matrices=False)

explained_variance = np.cumsum(s ** 2) / np.sum(s ** 2)
k = np.argmax(explained_variance >= 0.95) + 1  # Number of significant singular values


S_reduced = np.diag(s[:k])
U_reduced = U[:, :k]
Vt_reduced = Vt[:k, :]
data_reconstructed = U_reduced @ S_reduced @ Vt_reduced

residuals = np.linalg.norm(centered_data - data_reconstructed, axis=1)


threshold = np.mean(residuals) + 2 * np.std(residuals)
outliers = residuals > threshold


data_cleaned = data[~outliers]

print("Number of retained singular values", k)
print("Original data points:", len(data))
print("Detected outliers:", np.sum(outliers))
print("Remaining data points after outlier removal:", len(data_cleaned))

Number of retained singular values 54
Original data points: 31332
Detected outliers: 2215
Remaining data points after outlier removal: 29117


We know that 2000 fake data points were added to the matrix. In this version of the code we found that 95% of the variance in the dataset was captured by the top k=54 singular values. This accounted to 2215 data points being labelled as outliers and being removed from the dataset.

in the version of the code below, we do not find the number of singular values to keep based on the proportion of variance that they capture. Instead we choose k interatively and based on the number of data points that were removed.

We found that in this code,  choosing k equal to the k returned by the above code eliminates marginally fewer outliers.

In [ ]:

import numpy as np
import pandas as pd
from collections import Counter

data = pd.read_csv('data2.csv')
Z = data.to_numpy()


U, S, Vt = np.linalg.svd(Z, full_matrices=False)


k = 54
S_thresholded = np.zeros_like(S)
S_thresholded[:k] = S[:k]


Z_cleaned = (U * S_thresholded) @ Vt


diff = np.abs(Z - Z_cleaned)
row_norms = np.linalg.norm(diff, axis=1)

threshold = np.mean(row_norms) + 2 * np.std(row_norms)
outlier_indices = np.where(row_norms > threshold)[0]

num_outliers = len(outlier_indices)
print("Number of outliers:", num_outliers)



print("Indices of detected outliers:", outlier_indices)

Number of outliers: 2208
Indices of detected outliers: [   70    82   145 ... 31328 31330 31331]
